# Feature Engineering, Scaling & Encoding

**Module:** 1 — Foundations & Data Engineering  
**Week:** 3, Days 1-2  

## Learning Objectives
- Understand why feature engineering matters more than model selection
- Create numeric features: log transforms, binning, interactions, ratios
- Encode categorical variables: label, one-hot, ordinal, target encoding
- Scale/normalize features: StandardScaler, MinMaxScaler, RobustScaler
- Handle datetime features: cyclic encoding, time-based aggregations
- Build reusable feature pipelines with scikit-learn

## Resources
- [Kaggle Learn: Feature Engineering](https://www.kaggle.com/learn/feature-engineering)
- [scikit-learn Preprocessing docs](https://scikit-learn.org/stable/modules/preprocessing.html)
- [Feature Engineering for ML (Alice Zheng)](https://www.oreilly.com/library/view/feature-engineering-for/9781491953235/) — optional reading

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    LabelEncoder, OneHotEncoder, OrdinalEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from shared.viz_utils import setup_style
setup_style()

np.random.seed(42)

In [ ]:
# Create a rich dataset for feature engineering practice
n = 1000

df = pd.DataFrame({
    'age': np.random.randint(18, 75, n),
    'income': np.random.lognormal(10.5, 0.8, n).astype(int),
    'years_employed': np.random.randint(0, 35, n),
    'num_products': np.random.randint(1, 8, n),
    'credit_score': np.random.normal(650, 80, n).astype(int).clip(300, 850),
    'account_balance': np.random.lognormal(8, 1.5, n).round(2),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n,
                                   p=[0.30, 0.40, 0.20, 0.10]),
    'employment_type': np.random.choice(['Full-time', 'Part-time', 'Self-employed', 'Unemployed'], n,
                                         p=[0.55, 0.15, 0.20, 0.10]),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
    'signup_date': pd.date_range('2020-01-01', periods=n, freq='8h'),
    'last_login': pd.date_range('2024-01-01', periods=n, freq='2h'),
})

# Target: churn (somewhat correlated with features)
churn_prob = (
    0.3
    - 0.002 * df['num_products']
    + 0.005 * (df['age'] - 40).abs() / 40
    - 0.001 * df['credit_score'] / 100
    + 0.1 * (df['employment_type'] == 'Unemployed').astype(float)
).clip(0.05, 0.80)
df['churned'] = (np.random.random(n) < churn_prob).astype(int)

print(f"Shape: {df.shape}")
print(f"Churn rate: {df['churned'].mean():.1%}")
df.head()

---
## 1. Numeric Transformations

### Why Transform?
Many ML models (linear regression, SVM, neural nets) assume normally distributed features. Skewed data can hurt performance.

In [ ]:
# Visualize skewed distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flatten(), ['income', 'account_balance', 'age', 'credit_score']):
    df[col].hist(bins=40, ax=ax, edgecolor='black', alpha=0.7)
    skew = df[col].skew()
    ax.set_title(f'{col} (skew={skew:.2f})')

plt.suptitle('Raw Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Log transform — tames right-skewed distributions
df['income_log'] = np.log1p(df['income'])           # log1p = log(1 + x), handles zeros
df['balance_log'] = np.log1p(df['account_balance'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['income'], bins=40, alpha=0.7, label='Raw', edgecolor='black')
axes[0].set_title(f"Income Raw (skew={df['income'].skew():.2f})")

axes[1].hist(df['income_log'], bins=40, alpha=0.7, label='Log', edgecolor='black', color='orange')
axes[1].set_title(f"Income Log (skew={df['income_log'].skew():.2f})")

plt.tight_layout()
plt.show()

In [ ]:
# Binning — convert continuous to categorical (useful for non-linear relationships)
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 50, 65, 100],
                          labels=['18-25', '26-35', '36-50', '51-65', '65+'])

df['credit_tier'] = pd.qcut(df['credit_score'], q=4,
                              labels=['Poor', 'Fair', 'Good', 'Excellent'])

print("Age groups:")
print(df['age_group'].value_counts().sort_index())
print(f"\nCredit tiers (quartile-based):")
print(df['credit_tier'].value_counts().sort_index())

In [ ]:
# Interaction features — combine two features to capture joint effects
df['income_per_product'] = df['income'] / df['num_products']
df['balance_to_income'] = df['account_balance'] / (df['income'] + 1)
df['age_x_products'] = df['age'] * df['num_products']
df['employed_years_ratio'] = df['years_employed'] / (df['age'] - 17).clip(lower=1)

print("New features:")
df[['income_per_product', 'balance_to_income', 'age_x_products', 'employed_years_ratio']].describe().round(2)

In [ ]:
# Polynomial features — capture non-linear relationships
df['credit_score_sq'] = df['credit_score'] ** 2
df['age_sq'] = df['age'] ** 2

# Clipping — cap outliers
df['income_clipped'] = df['income'].clip(
    lower=df['income'].quantile(0.01),
    upper=df['income'].quantile(0.99)
)

### Numeric Transformation Cheat Sheet

| Technique | When to Use | Code |
|-----------|-------------|------|
| Log transform | Right-skewed data | `np.log1p(x)` |
| Square root | Moderate skew | `np.sqrt(x)` |
| Binning (equal-width) | Known meaningful thresholds | `pd.cut(x, bins=...)` |
| Binning (quantile) | No known thresholds | `pd.qcut(x, q=4)` |
| Ratios | Two related numerics | `x / y` |
| Interactions | Joint effects | `x * y` |
| Polynomial | Non-linear relationship | `x ** 2` |
| Clipping | Cap extreme outliers | `x.clip(lower, upper)` |

---
## 2. Categorical Encoding

In [ ]:
# Ordinal Encoding — when categories have a natural order
education_order = ['High School', 'Bachelor', 'Master', 'PhD']
df['education_ordinal'] = df['education'].map({v: i for i, v in enumerate(education_order)})

# Or with sklearn:
enc = OrdinalEncoder(categories=[education_order])
df['education_ordinal_sk'] = enc.fit_transform(df[['education']]).astype(int)

print("Ordinal encoding:")
df[['education', 'education_ordinal']].drop_duplicates().sort_values('education_ordinal')

In [ ]:
# One-Hot Encoding — when categories have NO natural order
# pd.get_dummies (quick and easy)
region_dummies = pd.get_dummies(df['region'], prefix='region', dtype=int)
print("One-hot encoded region:")
print(region_dummies.head())

# Drop first to avoid multicollinearity (for linear models)
region_dummies_dropped = pd.get_dummies(df['region'], prefix='region', drop_first=True, dtype=int)
print(f"\nWith drop_first (K-1 columns):")
print(region_dummies_dropped.head())

In [ ]:
# Target Encoding — replace category with mean of target variable
# CAUTION: Can leak information if not done properly (use within cross-validation)

# Simple target encoding (for demonstration — use sklearn's TargetEncoder in practice)
target_means = df.groupby('employment_type')['churned'].mean()
df['employment_target_enc'] = df['employment_type'].map(target_means)

print("Target encoding (mean churn rate per employment type):")
print(target_means.sort_values())

In [ ]:
# Frequency Encoding — replace category with its count/frequency
freq = df['region'].value_counts(normalize=True)
df['region_freq'] = df['region'].map(freq)

print("Frequency encoding:")
df[['region', 'region_freq']].drop_duplicates().sort_values('region_freq', ascending=False)

### Encoding Cheat Sheet

| Method | When to Use | # Columns | Tree Models | Linear Models |
|--------|-------------|-----------|-------------|---------------|
| Ordinal | Natural order (Low/Med/High) | 1 | Good | OK |
| One-Hot | No order, few categories (<10) | K or K-1 | OK | Good |
| Target | High cardinality | 1 | Good | Good |
| Frequency | High cardinality | 1 | Good | OK |
| Label | Only for tree models | 1 | Good | Bad |

---
## 3. Feature Scaling

In [ ]:
# Compare scaling methods
cols_to_scale = ['age', 'income', 'credit_score', 'account_balance']
X = df[cols_to_scale].copy()

scalers = {
    'StandardScaler (z-score)': StandardScaler(),
    'MinMaxScaler (0-1)': MinMaxScaler(),
    'RobustScaler (IQR)': RobustScaler(),
}

fig, axes = plt.subplots(len(scalers), len(cols_to_scale), figsize=(16, 10))

for i, (name, scaler) in enumerate(scalers.items()):
    scaled = pd.DataFrame(scaler.fit_transform(X), columns=cols_to_scale)
    for j, col in enumerate(cols_to_scale):
        axes[i][j].hist(scaled[col], bins=30, alpha=0.7, edgecolor='black')
        if j == 0:
            axes[i][j].set_ylabel(name, fontsize=10)
        if i == 0:
            axes[i][j].set_title(col)

plt.suptitle('Comparison of Scaling Methods', fontsize=14)
plt.tight_layout()
plt.show()

### When to Use Which Scaler

| Scaler | How It Works | Best For |
|--------|-------------|----------|
| **StandardScaler** | `(x - mean) / std` → mean=0, std=1 | Default choice, neural nets, SVM, PCA |
| **MinMaxScaler** | `(x - min) / (max - min)` → range [0, 1] | When you need bounded values, image data |
| **RobustScaler** | `(x - median) / IQR` | Data with outliers (uses median, not mean) |

**Tree-based models** (XGBoost, Random Forest) do NOT need scaling!

---
## 4. Datetime Features

In [ ]:
# Extract datetime components
df['signup_year'] = df['signup_date'].dt.year
df['signup_month'] = df['signup_date'].dt.month
df['signup_dayofweek'] = df['signup_date'].dt.dayofweek  # 0=Monday
df['signup_quarter'] = df['signup_date'].dt.quarter

# Time-based features
df['account_age_days'] = (df['last_login'] - df['signup_date']).dt.days
df['days_since_login'] = (pd.Timestamp('2024-12-31') - df['last_login']).dt.days

print("Datetime features:")
df[['signup_date', 'signup_year', 'signup_month', 'signup_dayofweek', 
    'account_age_days', 'days_since_login']].head()

In [ ]:
# Cyclic encoding — for features that wrap around (month, day of week, hour)
# Month 12 is close to month 1, but numerically they're far apart!
# Solution: encode as sin/cos pair

df['month_sin'] = np.sin(2 * np.pi * df['signup_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['signup_month'] / 12)

df['dow_sin'] = np.sin(2 * np.pi * df['signup_dayofweek'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['signup_dayofweek'] / 7)

# Visualize: months form a circle, not a line
fig, ax = plt.subplots(figsize=(6, 6))
months = range(1, 13)
month_sin = [np.sin(2 * np.pi * m / 12) for m in months]
month_cos = [np.cos(2 * np.pi * m / 12) for m in months]

ax.scatter(month_sin, month_cos, s=100, zorder=5)
for m, x, y in zip(months, month_sin, month_cos):
    ax.annotate(f"M{m}", (x, y), textcoords="offset points", xytext=(8, 8))

circle = plt.Circle((0, 0), 1, fill=False, linestyle='--', alpha=0.3)
ax.add_patch(circle)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_xlabel('sin(month)')
ax.set_ylabel('cos(month)')
ax.set_title('Cyclic Encoding: Months on a Circle')
ax.set_aspect('equal')
plt.show()

---
## 5. Sklearn Pipelines — Putting It All Together

Build a reusable preprocessing pipeline using `ColumnTransformer` and `Pipeline`.

In [ ]:
from sklearn.impute import SimpleImputer

# Define column groups
numeric_features = ['age', 'income', 'credit_score', 'account_balance', 
                     'years_employed', 'num_products']
ordinal_features = ['education']
nominal_features = ['employment_type', 'region']

# Build transformers for each group
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master', 'PhD']])),
])

nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')),
])

# Combine with ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('nom', nominal_transformer, nominal_features),
    ],
    remainder='drop'  # Drop columns not specified
)

# Fit and transform
X_raw = df[numeric_features + ordinal_features + nominal_features]
X_processed = preprocessor.fit_transform(X_raw)

# Get feature names
feature_names = (numeric_features + 
                 ['education_ordinal'] +
                 list(preprocessor.named_transformers_['nom']
                      .named_steps['encoder']
                      .get_feature_names_out(nominal_features)))

X_df = pd.DataFrame(X_processed, columns=feature_names)
print(f"Processed shape: {X_df.shape}")
print(f"Feature names: {feature_names}")
X_df.head()

In [ ]:
# The pipeline can be used in a model pipeline!
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Full pipeline: preprocess → model
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000)),
])

y = df['churned']
scores = cross_val_score(full_pipeline, X_raw, y, cv=5, scoring='accuracy')
print(f"Cross-validated accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

### Why Pipelines?
1. **No data leakage** — scaler fits only on training data during CV
2. **Reproducible** — same transform applied to train and test
3. **Deployable** — serialize the entire pipeline with `joblib`
4. **Clean** — no manual intermediate steps

---
## 6. Feature Selection Preview

Quick look at which features matter — we'll go deeper in Module 2.

In [ ]:
# Correlation with target
feature_cols = ['age', 'income', 'credit_score', 'account_balance', 
                'years_employed', 'num_products', 'education_ordinal',
                'income_per_product', 'balance_to_income', 'account_age_days']

correlations = df[feature_cols + ['churned']].corr()['churned'].drop('churned').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
correlations.plot.barh(ax=ax, edgecolor='black')
ax.set_title('Feature Correlation with Churn')
ax.set_xlabel('Pearson Correlation')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

---
## 7. Practice Exercises

### Exercise 1: Creative Features
Using the same dataset, create 5 additional features that you think could help predict churn. For each, explain your reasoning.

In [ ]:
# Exercise 1: Your 5 new features


### Exercise 2: Build a Complete Preprocessor
Create a `ColumnTransformer` that:
- Log-transforms `income` and `account_balance` before scaling
- Ordinal encodes `education` and `credit_tier`
- One-hot encodes `region` and `employment_type`
- Includes `account_age_days` as a numeric feature

Verify it works with `cross_val_score` and a LogisticRegression.

In [ ]:
# Exercise 2: Your pipeline


### Exercise 3: Impact Comparison
Train a LogisticRegression with:
1. Only the original 6 numeric features (scaled)
2. All original features + your engineered features

Compare cross-validated accuracy. How much did feature engineering help?

In [ ]:
# Exercise 3: Compare


---
## Key Takeaways

1. **Feature engineering often matters more than model choice** — good features with a simple model beat bad features with a complex model.
2. **Log transforms** are your best friend for skewed data (income, prices, counts).
3. **Ratios and interactions** capture relationships between variables that models might miss.
4. **Encoding choice depends on the model** — one-hot for linear, ordinal/target for trees.
5. **Scaling matters for distance-based models** (linear, SVM, KNN, neural nets) but NOT for tree models.
6. **Use sklearn Pipelines** to prevent data leakage and keep preprocessing reproducible.
7. **Cyclic encoding** (sin/cos) is essential for circular features like month, hour, day of week.